### Overview
- We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

### Installation

In [1]:
# !pip install langchain-core langgraph>0.2.27

In [2]:
import os
api_key = os.getenv("TOGETHERAI_API_KEY")

In [3]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    base_url="https://api.together.xyz/v1",
    api_key=api_key,
    model='meta-llama/Llama-3.3-70B-Instruct-Turbo-Free',
)

In [4]:
from langchain_core.messages import HumanMessage

model.invoke([HumanMessage(content="Hi! I'm Bob and I am going to start gym from Monday ")])

AIMessage(content="Hi Bob! Congratulations on taking the first step towards a healthier lifestyle by joining the gym. Starting on Monday is a great idea, it's a fresh start to the week. What are your fitness goals, and what motivated you to start going to the gym?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 49, 'total_tokens': 102, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'cached_tokens': 0}, 'model_name': 'meta-llama/Llama-3.3-70B-Instruct-Turbo-Free', 'system_fingerprint': None, 'id': 'o3WSCmA-62bZhn-960e8ab2fdb06efa', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--469cd0f6-3e25-4029-9a7f-9df7a6e1b017-0', usage_metadata={'input_tokens': 49, 'output_tokens': 53, 'total_tokens': 102, 'input_token_details': {}, 'output_token_details': {}})

In [5]:
model.invoke([HumanMessage(content="Hi! I'm Bob and I am going to start gym from Monday ")]).content

"Hi Bob! Congratulations on taking the first step towards a healthier lifestyle by joining the gym. Starting on Monday is a great idea, it's a fresh start to the week. What are your fitness goals, and what motivated you to start going to the gym?"

``The model on its own does not have any concept of state. For example, if you ask a followup question:``

In [6]:
model.invoke([HumanMessage(content="What's my name?")]).content

"I don't know your name. I'm a large language model, I don't have any information about you, including your name. I'm here to help answer your questions and provide assistance, but I don't have any personal knowledge about you. Would you like to introduce yourself?"

## Message persistence

In [7]:
# !pip install langgraph


In [8]:
# Import necessary components from langgraph library
from langgraph.checkpoint.memory import MemorySaver  # MemorySaver is used to save the state of the workflow
from langgraph.graph import START, MessagesState, StateGraph  # Import graph components for workflow and state management

- **MemorySaver**: Used to save and manage the workflow state during execution. This enables checkpointing and recovery if needed.
- **START**: A predefined starting point in the graph.
- **MessagesState**: A schema that defines the structure of the state passed between nodes (likely contains the messages key).
- **StateGraph**: A class used to define a graph-based workflow where nodes represent tasks, and edges define the flow

In [9]:

# Define a new graph object, using the MessagesState schema to structure the state for the graph
workflow = StateGraph(state_schema=MessagesState)

- Creates a new instance of StateGraph called workflow.
- state_schema=MessagesState: Specifies that the state passed through the graph will follow the MessagesState schema, ensuring consistency in the state structure.


In [10]:

# Define the function that calls the model
def call_model(state: MessagesState):
    # The model is invoked with the 'messages' key from the state and the response is returned
    response = model.invoke(state["messages"])
    # Return the model's response as a dictionary with the 'messages' key
    return {"messages": response}

- workflow.add_edge(START, "model"):
- Defines the transition (edge) from the starting point (START) to the "model" node.
    This sets up the execution flow of the graph.
    workflow.add_node("model", call_model):
    Adds the "model" node to the graph.
    Associates the call_model function with this node, meaning that when the "model" node is activated, it will execute call_model.

In [11]:

# Define the (single) node in the graph:
# The 'model' node in the workflow calls the 'call_model' function when activated
workflow.add_edge(START, "model")  # Create an edge from the starting point (START) to the 'model' node
workflow.add_node("model", call_model)  # Add the 'model' node and associate it with the 'call_model' function

In [12]:



# Initialize memory to store state information during the graph's execution
memory = MemorySaver()  # MemorySaver will manage storing and retrieving memory at each step of the graph's execution

# Compile the workflow into a callable application, passing in the memory as a checkpoint
app = workflow.compile(checkpointer=memory)  # The workflow is compiled into an app with memory checkpointing enabled


In [19]:
config = {"configurable": {"thread_id": "abc123"}}

In [20]:
query = "Hi! I'm Bob."

input_messages = [HumanMessage(query)]
output = app.invoke({"messages": input_messages}, config)
output["messages"][-1].pretty_print()  # output contains all messages in state

================================== Ai Message ==================================

Hello Bob! It's nice to meet you. Is there something I can help you with or would you like to chat?


In [21]:
output

{'messages': [HumanMessage(content="Hi! I'm Bob.", additional_kwargs={}, response_metadata={}, id='6bc27807-bc68-4cd9-9733-5d16d0e33d69'),
  AIMessage(content="Hello Bob! It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 41, 'total_tokens': 67, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'cached_tokens': 0}, 'model_name': 'meta-llama/Llama-3.3-70B-Instruct-Turbo-Free', 'system_fingerprint': None, 'id': 'o3WTWsF-62bZhn-960e90f46e27423c', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--30f37137-e58f-49fc-b9f6-da28edd04efc-0', usage_metadata={'input_tokens': 41, 'output_tokens': 26, 'total_tokens': 67, 'input_token_details': {}, 'output_token_details': {}})]}

In [22]:
query = "What's my name?"

input_messages = [HumanMessage(query)]
output = app.invoke({"messages": input_messages}, config)
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

Your name is Bob. You told me that when you introduced yourself.


In [23]:
output

{'messages': [HumanMessage(content="Hi! I'm Bob.", additional_kwargs={}, response_metadata={}, id='6bc27807-bc68-4cd9-9733-5d16d0e33d69'),
  AIMessage(content="Hello Bob! It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 41, 'total_tokens': 67, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'cached_tokens': 0}, 'model_name': 'meta-llama/Llama-3.3-70B-Instruct-Turbo-Free', 'system_fingerprint': None, 'id': 'o3WTWsF-62bZhn-960e90f46e27423c', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--30f37137-e58f-49fc-b9f6-da28edd04efc-0', usage_metadata={'input_tokens': 41, 'output_tokens': 26, 'total_tokens': 67, 'input_token_details': {}, 'output_token_details': {}}),
  HumanMessage(content="What's my name?", additional_kwargs={}, response_metadata={}, id='80a51363-2a47-4883-bfd9-bf729c6d143d'),
  

In [24]:
query = "I met jane my freind yesterday and she looked upset "

input_messages = [HumanMessage(query)]
output = app.invoke({"messages": input_messages}, config)
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

It sounds like you're a bit concerned about your friend Jane. What happened when you met her yesterday? Did she tell you what was wrong or what was upsetting her?


In [25]:
query = "what is the name of my freind and her husband name "

input_messages = [HumanMessage(query)]
output = app.invoke({"messages": input_messages}, config)
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

You mentioned that your friend's name is Jane, but you didn't mention her husband's name. You only told me about meeting Jane and that she looked upset, but you didn't share any information about her husband. Would you like to share more about him?


In [26]:
output

{'messages': [HumanMessage(content="Hi! I'm Bob.", additional_kwargs={}, response_metadata={}, id='6bc27807-bc68-4cd9-9733-5d16d0e33d69'),
  AIMessage(content="Hello Bob! It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 41, 'total_tokens': 67, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'cached_tokens': 0}, 'model_name': 'meta-llama/Llama-3.3-70B-Instruct-Turbo-Free', 'system_fingerprint': None, 'id': 'o3WTWsF-62bZhn-960e90f46e27423c', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--30f37137-e58f-49fc-b9f6-da28edd04efc-0', usage_metadata={'input_tokens': 41, 'output_tokens': 26, 'total_tokens': 67, 'input_token_details': {}, 'output_token_details': {}}),
  HumanMessage(content="What's my name?", additional_kwargs={}, response_metadata={}, id='80a51363-2a47-4883-bfd9-bf729c6d143d'),
  

In [27]:
len("Hi Bob! It's nice to meet you. Is there something I can help you with or would you like to chat?".split(" "))

21

### NEW CONFIGUERATION

In [ ]:
config = {"configurable": {"thread_id": "abc234"}}
query= 'what is my freinds name'

input_messages = [HumanMessage(query)]
output = app.invoke({"messages": input_messages}, config)
output["messages"][-1].pretty_print()

In [ ]:
config = {"configurable": {"thread_id": "abc234"}}
query= 'I am in jodhpur to teach ai and ml to students. suggest me one line to motivate them'

input_messages = [HumanMessage(query)]
output = app.invoke({"messages": input_messages}, config)
output["messages"][-1].pretty_print()